<a href="https://colab.research.google.com/github/thanujayarabolu/24B11CS501_DriverReady_OPC_2026/blob/main/AgentAI_DAY_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
%pip install -qU langchain langchain-groq

In [18]:
from langchain_groq import ChatGroq
from google.colab import userdata

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3,
    api_key=userdata.get("GROQ_API_KEY")
)

print("Travel Buddy AI Assistant is ready!")

Travel Buddy AI Assistant is ready!


In [19]:
SYSTEM = """
You are Travel Buddy, a friendly AI travel assistant.

Your job is to help customers with travel packages,
destinations, prices, and trip information.

Sample travel package details:
1. Araku Valley 2 Days / 1 Night package: Rs. 4999 per person.
2. Vizag Local 1 Day package: Rs. 1499 per person.
3. Hyderabad 2 Days / 1 Night package: Rs. 3999 per person.

Rules:
- Give simple and friendly answers.
- Mention prices only from the sample package details.
- For booking availability, say "Please check with our travel team."
- Never promise a booking without confirmation.
- Never invent travel schedules or availability.
- If you don't know something, say so honestly.
- Ask customers for destination and travel date when needed.
"""

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("placeholder", "{chat_history}"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("Travel Buddy chain created successfully!")

Travel Buddy chain created successfully!


In [21]:
chat_history = []

def ask_travel_buddy(question):
    response = chain.invoke({
        "question": question,
        "chat_history": chat_history
    })

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))

    return response

print(ask_travel_buddy("What is the price of Araku package?"))

The Araku Valley 2‑Day / 1‑Night package is priced at **₹4,999 per person**.


In [22]:
print(ask_travel_buddy("What is included in the package?"))
print(ask_travel_buddy("Do you have family packages?"))
print(ask_travel_buddy("Can I book for tomorrow?"))

I’m sorry, but I don’t have the specific details of what’s included in the Araku Valley package. The price is ₹4,999 per person. If you’d like more information or help planning a trip, feel free to let me know the destination and travel dates, and I’ll do my best to assist!
I’m not aware of any specific family‑only packages in our current list. We do offer the Araku Valley 2‑Day / 1‑Night package at ₹4,999 per person, the Vizag Local 1‑Day at ₹1,499, and the Hyderabad 2‑Day / 1‑Night at ₹3,999.

If you’re looking for something tailored for families, could you let me know the destination and travel dates you have in mind? I can check with our travel team and see what options might suit a family group.
I’m not able to confirm availability directly. Please check with our travel team for booking tomorrow.


In [23]:
import pandas as pd

tests = [
    {
        "question": "How much is the Araku 2-day package?",
        "must_include": "4999"
    },
    {
        "question": "Do you have family travel packages?",
        "must_include": "family"
    },
    {
        "question": "Can I book for tomorrow?",
        "must_include": "check"
    },
    {
        "question": "Do you arrange trips to Hyderabad?",
        "must_include": "Hyderabad"
    },
    {
        "question": "What is the price of Vizag local package?",
        "must_include": "1499"
    }
]
df = pd.DataFrame(tests)
df.to_csv("tests.csv", index=False)

print(df)

                                    question must_include
0       How much is the Araku 2-day package?         4999
1        Do you have family travel packages?       family
2                   Can I book for tomorrow?        check
3         Do you arrange trips to Hyderabad?    Hyderabad
4  What is the price of Vizag local package?         1499


In [11]:
import csv

def run_tests(path="/content/tests.csv"):
    with open(path, "r") as file:
        rows = list(csv.DictReader(file))
    passed = 0
    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })
        expected = row["must_include"]
        ok = expected.lower() in answer.lower()
        if ok:
            passed += 1
        print("PASS" if ok else "FAIL",
              "|", row["question"])
        print("Answer:", answer)
        print("-" * 50)
    print(f"\n{passed}/{len(rows)} tests passed")
    print(f"Success Rate: {passed/len(rows)*100:.0f}%")

run_tests()

PASS | How much is a plum cake?
Answer: A plum cake costs **Rs 500**.  
Please note that we require a **24‑hour notice** for cake orders.
--------------------------------------------------
PASS | Do you make eggless cakes?
Answer: Yes, all our cakes are eggless.
--------------------------------------------------
FAIL | Can I get a cake today evening?
Answer: I’m sorry, but we can’t take same‑day cake orders. All cake orders require a 24‑hour notice, so we’d need to schedule it in advance. Let me know the date and time you’d like, and I’ll check availability for you.
--------------------------------------------------
PASS | Do you deliver to Kakinada?
Answer: I’m not sure if we can deliver to Kakinada at the moment. Let me check with the owner and get back to you shortly.
--------------------------------------------------
PASS | What is the price of 6 cupcakes?
Answer: The price for 6 cupcakes is Rs 480.
--------------------------------------------------

4/5 tests passed
Success Rate: 

In [24]:
results = []

for index, row in df.iterrows():
    answer = ask_travel_buddy(row["question"])
    passed = row["must_include"].lower() in answer.lower()
    results.append({
        "Question": row["question"],
        "Expected": row["must_include"],
        "Answer": answer,
        "Result": "PASS" if passed else "FAIL"
    })
results_df = pd.DataFrame(results)
display(results_df)

pass_count = sum(
    result["Result"] == "PASS"
    for result in results
)
print("Pass Rate:", pass_count, "/", len(results))

,Question,Expected,Answer,Result
0,How much is the Araku 2-day package?,4999,The Araku Valley 2‑Day / 1‑Night package is **...,FAIL
1,Do you have family travel packages?,family,I don’t see any dedicated family‑only packages...,PASS
2,Can I book for tomorrow?,check,I’m not able to confirm availability directly....,PASS
3,Do you arrange trips to Hyderabad?,Hyderabad,Absolutely! We have a **Hyderabad 2‑Day / 1‑Ni...,PASS
4,What is the price of Vizag local package?,1499,"The Vizag Local 1‑Day package is **₹1,499 per ...",FAIL


Pass Rate: 3 / 5


In [25]:
SYSTEM = SYSTEM + """

Additional instructions:
- Always explain that package prices are sample prices.
- Never confirm a booking without travel team approval.
- If the customer asks about availability,
  request their travel date and destination.
- Keep answers clear and short.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("placeholder", "{chat_history}"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("Travel Buddy prompt improved!")

Travel Buddy prompt improved!


In [26]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0,
    api_key=userdata.get("GROQ_API_KEY")
)

chain = prompt | llm | StrOutputParser()

print(ask_travel_buddy("Suggest a trip to Araku Valley."))

Sure!  
- **Araku Valley 2‑Day / 1‑Night package** – sample price **₹4,999 per person**  

We can customize the itinerary to your liking (transport, meals, sightseeing spots).  
Could you let me know your preferred travel dates so I can check availability for you?


In [27]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Suggest a travel package to Araku"}
    ]
})

print(result["messages"][-1].content)

Here’s a great option for a quick getaway to Araku Valley:

**Araku Valley – 2 Days / 1 Night Package**  
*Sample price: Rs. 4,999 per person*

This includes round‑trip travel from Hyderabad (or your nearest city), a guided tour of the scenic hills, and a comfortable stay at a mid‑range resort.

**Next steps:**  
Please check with our travel team for availability on your preferred dates. If you let us know the dates you’re planning to travel, we can help you find the best option!


In [28]:
def get_package_price(destination: str) -> str:
    """
    Returns the sample travel package price
    for the given destination.
    """

    packages = {
        "araku": 4999,
        "vizag": 1499,
        "hyderabad": 3999
    }

    price = packages.get(destination.lower())

    if price is None:
        return "Package price not available. Please check with our team."

    return f"Sample package price for {destination}: Rs. {price}"

def check_trip_availability(destination: str, travel_date: str) -> str:
    """
    Checks whether trip availability has been confirmed.
    This demo does not connect to a real booking system.
    """

    return (
        f"Availability for {destination} on {travel_date} "
        "must be confirmed by the travel team."
    )

print(get_package_price("Araku"))
print(check_trip_availability("Araku", "2026-12-10"))

Sample package price for Araku: Rs. 4999
Availability for Araku on 2026-12-10 must be confirmed by the travel team.
